In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("Cars.csv")

print("Shape:", df.shape)
df.head()

Shape: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [4]:
# Same preprocessing used in A2

df = df.dropna().copy()

# Map owner categories
owner_mapping = {
    'First Owner': 1,
    'Second Owner': 2,
    'Third Owner': 3,
    'Fourth & Above Owner': 4,
    'Test Drive Car': 5
}

df['owner'] = df['owner'].map(owner_mapping)

# Extract brand from car name
df['brand'] = df['name'].str.split().str[0]

# Drop original name column
df = df.drop('name', axis=1)

# Convert numerical columns to numeric values
for col in ['mileage', 'engine', 'max_power', 'seats']:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0],
        errors='coerce'
    )

# Remove rows made invalid by conversion
df = df.dropna().copy()

print("Cleaned shape:", df.shape)

Cleaned shape: (7906, 13)


In [5]:
# Convert selling_price into 4 classification classes
df["price_class"] = pd.qcut(
    df["selling_price"],
    q=4,
    labels=[0, 1, 2, 3]
).astype(int)

print(df["price_class"].value_counts().sort_index())

price_class
0    2031
1    1949
2    1963
3    1963
Name: count, dtype: int64


In [6]:
import time
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------------------
# Task 2: regularization classes (same idea as the Ridge class in Regularization.ipynb)
# ----------------------------------------------------------------------------------
class Ridge:
    """L2 penalty:  lambda * sum(theta^2).

    The bias row (row 0 of W, because we prepend a column of ones to X) is NOT penalised.
    The assignment formula sums j from 1 to n, so theta_0 is left out.
    Shrinking the intercept would only push predictions toward a fixed value
    and does nothing to reduce the model's complexity.
    """

    def __init__(self, l):
        self.l = l

    def __call__(self, W):
        # penalty added to the loss; W shape (n, k), skip the bias row W[0]
        return self.l * np.sum(np.square(W[1:]))

    def derivation(self, W):
        # d/dW of lambda * W^2 = 2 * lambda * W ; bias row gets zero penalty gradient
        grad = 2 * self.l * W
        grad[0] = 0.0
        return grad


class NoPenalty:
    """Used when use_ridge=False, so the training loop never needs an if-statement."""

    def __call__(self, W):
        return 0.0

    def derivation(self, W):
        return np.zeros_like(W)


# ----------------------------------------------------------------------------------
# The model
# ----------------------------------------------------------------------------------
class LogisticRegression:
    """Multinomial logistic regression (softmax) trained with gradient descent.

    Parameters
    ----------
    k          : number of classes
    n          : number of features INCLUDING the intercept column of ones
    method     : 'batch', 'minibatch' or 'sto'
    alpha      : learning rate
    max_iter   : number of parameter updates
    use_ridge  : if True, add the L2 penalty (Task 2)
    l          : lambda, the strength of the L2 penalty (ignored when use_ridge=False)
    batch_frac : fraction of the training rows used per mini-batch step
    verbose    : print the loss every 500 iterations
    """

    def __init__(self, k, n, method="batch", alpha=0.001, max_iter=5000,
                 use_ridge=False, l=0.01, batch_frac=0.3, verbose=True):
        self.k = k
        self.n = n
        self.alpha = alpha
        self.max_iter = max_iter
        self.method = method
        self.use_ridge = use_ridge
        self.l = l
        self.batch_frac = batch_frac
        self.verbose = verbose
        # pick the penalty object once; both share the same interface
        self.regularization = Ridge(l) if use_ridge else NoPenalty()

    # ------------------------------------------------------------------ training
    def fit(self, X, Y):
        """X shape (m, n) with the intercept column, Y shape (m, k) one-hot."""
        self.W = np.random.rand(self.n, self.k)   # random start, same as the lecture
        self.losses = []
        m = X.shape[0]
        start_time = time.time()

        if self.method not in ("batch", "minibatch", "sto"):
            raise ValueError('Method must be one of the followings: "batch", "minibatch" or "sto".')

        batch_size = max(1, int(self.batch_frac * m))

        for i in range(self.max_iter):
            if self.method == "batch":
                # every sample is used in every step
                X_b, Y_b = X, Y
            elif self.method == "minibatch":
                # a random subset of rows (no repeats inside one batch)
                ix = np.random.choice(m, size=batch_size, replace=False)
                X_b, Y_b = X[ix], Y[ix]
            else:  # 'sto'
                # exactly one random row per step; reshape keeps the 2-D shape (1, n) / (1, k)
                ix = np.random.randint(m)
                X_b, Y_b = X[ix:ix + 1], Y[ix:ix + 1]

            loss, grad = self.gradient(X_b, Y_b)
            self.losses.append(loss)
            self.W = self.W - self.alpha * grad   # gradient descent update

            if self.verbose and i % 500 == 0:
                print(f"Loss at iteration {i}", loss)

        self.train_time = time.time() - start_time
        if self.verbose:
            print(f"time taken: {self.train_time}")
        return self

    def gradient(self, X, Y):
        """Returns (loss, gradient) for the given rows.

        loss = -(1/m) * sum(Y * log(h))  +  lambda * sum(W[1:]^2)
        grad = (1/m) * X^T (h - Y)       +  2 * lambda * W  (bias row excluded)
        """
        m = X.shape[0]
        h = self.h_theta(X, self.W)
        # clip so log never sees exactly 0
        cross_entropy = -np.sum(Y * np.log(np.clip(h, 1e-15, 1.0))) / m
        loss = cross_entropy + self.regularization(self.W)

        error = h - Y                                   # (m, k)
        grad = self.softmax_grad(X, error) / m          # (n, k), averaged like the loss
        grad = grad + self.regularization.derivation(self.W)
        return loss, grad

    def softmax(self, theta_t_x):
        # subtracting the row max does not change the result but avoids exp() overflow
        z = theta_t_x - np.max(theta_t_x, axis=1, keepdims=True)
        e = np.exp(z)
        return e / np.sum(e, axis=1, keepdims=True)

    def softmax_grad(self, X, error):
        return X.T @ error

    def h_theta(self, X, W):
        """X (m, n) @ W (n, k) -> probabilities (m, k), each row sums to 1."""
        return self.softmax(X @ W)

    # ------------------------------------------------------------------ prediction
    def predict_proba(self, X):
        return self.h_theta(X, self.W)

    def predict(self, X_test):
        """Class with the highest probability for every row -> shape (m,)."""
        return np.argmax(self.h_theta(X_test, self.W), axis=1)

    def plot(self):
        plt.plot(np.arange(len(self.losses)), self.losses, label="Train Losses")
        plt.title("Losses")
        plt.xlabel("iteration")
        plt.ylabel("losses")
        plt.legend()

    # ------------------------------------------------------------------ Task 1: metrics
    # Every metric below works on class LABELS (shape (m,)), not one-hot.
    # Per class c:  TP = predicted c and really c
    #               FP = predicted c but really something else
    #               FN = really c but predicted something else

    def _counts(self, y_true, y_pred, c):
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        return tp, fp, fn

    def accuracy(self, y_true, y_pred):
        """correct predictions / all predictions"""
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        return np.sum(y_true == y_pred) / len(y_true)

    def precision(self, y_true, y_pred, c):
        """TP / (TP + FP) for class c. If the class is never predicted we return 0
        (the same convention as sklearn's default zero_division)."""
        tp, fp, _ = self._counts(y_true, y_pred, c)
        return tp / (tp + fp) if (tp + fp) > 0 else 0.0

    def recall(self, y_true, y_pred, c):
        """TP / (TP + FN) for class c. 0 if the class never appears in y_true."""
        tp, _, fn = self._counts(y_true, y_pred, c)
        return tp / (tp + fn) if (tp + fn) > 0 else 0.0

    def f1_score(self, y_true, y_pred, c):
        """2 * precision * recall / (precision + recall) for class c."""
        p = self.precision(y_true, y_pred, c)
        r = self.recall(y_true, y_pred, c)
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    def support(self, y_true, c):
        """How many samples of class c are in the TRUE labels."""
        return int(np.sum(np.asarray(y_true) == c))

    # ---- macro averages: every class counts the same
    def macro_precision(self, y_true, y_pred):
        return np.mean([self.precision(y_true, y_pred, c) for c in range(self.k)])

    def macro_recall(self, y_true, y_pred):
        return np.mean([self.recall(y_true, y_pred, c) for c in range(self.k)])

    def macro_f1(self, y_true, y_pred):
        return np.mean([self.f1_score(y_true, y_pred, c) for c in range(self.k)])

    # ---- weighted averages: each class is weighted by its share of the true labels
    def _weights(self, y_true):
        sup = np.array([self.support(y_true, c) for c in range(self.k)])
        return sup / sup.sum()

    def weighted_precision(self, y_true, y_pred):
        w = self._weights(y_true)
        return np.sum(w * np.array([self.precision(y_true, y_pred, c) for c in range(self.k)]))

    def weighted_recall(self, y_true, y_pred):
        w = self._weights(y_true)
        return np.sum(w * np.array([self.recall(y_true, y_pred, c) for c in range(self.k)]))

    def weighted_f1(self, y_true, y_pred):
        w = self._weights(y_true)
        return np.sum(w * np.array([self.f1_score(y_true, y_pred, c) for c in range(self.k)]))

    def classification_report(self, y_true, y_pred):
        """Prints a report that looks like sklearn's classification_report and
        returns the same numbers as a dict so they can be compared or logged."""
        report = {}
        lines = [f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>10}"]
        for c in range(self.k):
            p, r, f = (self.precision(y_true, y_pred, c),
                       self.recall(y_true, y_pred, c),
                       self.f1_score(y_true, y_pred, c))
            s = self.support(y_true, c)
            report[str(c)] = {"precision": p, "recall": r, "f1-score": f, "support": s}
            lines.append(f"{c:>14}{p:>10.2f}{r:>10.2f}{f:>10.2f}{s:>10}")

        total = len(y_true)
        acc = self.accuracy(y_true, y_pred)
        report["accuracy"] = acc
        lines.append("")
        lines.append(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.2f}{total:>10}")

        macro = (self.macro_precision(y_true, y_pred), self.macro_recall(y_true, y_pred),
                 self.macro_f1(y_true, y_pred))
        weighted = (self.weighted_precision(y_true, y_pred), self.weighted_recall(y_true, y_pred),
                    self.weighted_f1(y_true, y_pred))
        report["macro avg"] = dict(zip(["precision", "recall", "f1-score"], macro), support=total)
        report["weighted avg"] = dict(zip(["precision", "recall", "f1-score"], weighted), support=total)
        lines.append(f"{'macro avg':>14}{macro[0]:>10.2f}{macro[1]:>10.2f}{macro[2]:>10.2f}{total:>10}")
        lines.append(f"{'weighted avg':>14}{weighted[0]:>10.2f}{weighted[1]:>10.2f}{weighted[2]:>10.2f}{total:>10}")
        print("\n".join(lines))
        return report


def one_hot(y, k):
    """Class labels (m,) -> one-hot matrix (m, k). Same idea as the lecture's for-loop,
    written as one line: row i of np.eye(k) has a 1 in column y[i]."""
    return np.eye(k)[np.asarray(y).astype(int)]

In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# features come from df (not the already dummy-encoded X), torque is dropped as in A2
X = df.drop(columns=["selling_price", "price_class", "torque"])
y = df["price_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["year", "km_driven", "mileage", "engine", "max_power", "seats", "owner"]
categorical_features = ["brand", "fuel", "seller_type", "transmission"]

# scale numerics, one-hot the categoricals
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

def add_intercept(A):
    return np.column_stack([np.ones(len(A)), A])

# fit on TRAIN only, then apply the same fitted transformer to test
X_train_model = add_intercept(preprocessor.fit_transform(X_train))
X_test_model = add_intercept(preprocessor.transform(X_test))
Y_train_onehot = np.eye(4)[y_train.astype(int)]

print(X_train_model.shape, X_train_model.dtype)

(6324, 47) float64


In [8]:
np.random.seed(42)   # same random starting weights every run, so results can be reproduced

model = LogisticRegression(
    k=4,
    n=X_train_model.shape[1],
    method="batch",
    alpha=0.1,          # was 0.001: too small now that the features are scaled
    max_iter=2000,
    use_ridge=False,
    verbose=False
)

model.fit(X_train_model, Y_train_onehot)

print("Model training completed.")
print("Weight shape:", model.W.shape)

Model training completed.
Weight shape: (47, 4)


In [11]:
# Predict on the test set
y_pred = model.predict(X_test_model)

print("First 20 predictions:")
print(y_pred[:20])

print("\nFirst 20 actual labels:")
print(y_test[:20])

print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

First 20 predictions:
[1 2 1 0 0 3 3 2 1 2 3 1 1 0 0 0 1 2 2 3]

First 20 actual labels:
[0 1 1 0 0 3 3 2 1 2 1 0 1 0 0 0 0 2 1 3]

Test Accuracy:
0.7446270543615676


In [12]:
[name for name in globals() if "report" in name.lower()]

['classification_report']

In [13]:
def custom_classification_report(y_true, y_pred, labels=None):
    if labels is None:
        labels = np.unique(y_true)

    report = {}

    # Per-class metrics
    for label in labels:
        tp = np.sum((y_true == label) & (y_pred == label))
        fp = np.sum((y_true != label) & (y_pred == label))
        fn = np.sum((y_true == label) & (y_pred != label))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )
        support = np.sum(y_true == label)

        report[int(label)] = {
            "precision": precision,
            "recall": recall,
            "f1-score": f1,
            "support": support
        }

    # Macro averages
    precisions = [report[int(label)]["precision"] for label in labels]
    recalls = [report[int(label)]["recall"] for label in labels]
    f1_scores = [report[int(label)]["f1-score"] for label in labels]
    supports = [report[int(label)]["support"] for label in labels]

    report["macro avg"] = {
        "precision": np.mean(precisions),
        "recall": np.mean(recalls),
        "f1-score": np.mean(f1_scores),
        "support": np.sum(supports)
    }

    # Weighted averages
    total_support = np.sum(supports)

    report["weighted avg"] = {
        "precision": np.average(precisions, weights=supports),
        "recall": np.average(recalls, weights=supports),
        "f1-score": np.average(f1_scores, weights=supports),
        "support": total_support
    }

    return report


custom_report = custom_classification_report(
    y_test,
    y_pred,
    labels=np.array([0, 1, 2, 3])
)

custom_report

{0: {'precision': np.float64(0.851010101010101),
  'recall': np.float64(0.8300492610837439),
  'f1-score': np.float64(0.8403990024937655),
  'support': np.int64(406)},
 1: {'precision': np.float64(0.6417525773195877),
  'recall': np.float64(0.6384615384615384),
  'f1-score': np.float64(0.6401028277634961),
  'support': np.int64(390)},
 2: {'precision': np.float64(0.65),
  'recall': np.float64(0.6946564885496184),
  'f1-score': np.float64(0.6715867158671587),
  'support': np.int64(393)},
 3: {'precision': np.float64(0.843915343915344),
  'recall': np.float64(0.811704834605598),
  'f1-score': np.float64(0.827496757457847),
  'support': np.int64(393)},
 'macro avg': {'precision': np.float64(0.7466695055612582),
  'recall': np.float64(0.7437180306751245),
  'f1-score': np.float64(0.7448963258955669),
  'support': np.int64(1582)},
 'weighted avg': {'precision': np.float64(0.7477258763106639),
  'recall': np.float64(0.7446270543615676),
  'f1-score': np.float64(0.7458798374569908),
  'suppor

In [14]:
sklearn_report = classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3],
    output_dict=True
)

print("Sklearn classification report:")
print(sklearn_report)

Sklearn classification report:
{'0': {'precision': 0.851010101010101, 'recall': 0.8300492610837439, 'f1-score': 0.8403990024937655, 'support': 406.0}, '1': {'precision': 0.6417525773195877, 'recall': 0.6384615384615384, 'f1-score': 0.6401028277634961, 'support': 390.0}, '2': {'precision': 0.65, 'recall': 0.6946564885496184, 'f1-score': 0.6715867158671587, 'support': 393.0}, '3': {'precision': 0.843915343915344, 'recall': 0.811704834605598, 'f1-score': 0.8274967574578469, 'support': 393.0}, 'accuracy': 0.7446270543615676, 'macro avg': {'precision': 0.7466695055612582, 'recall': 0.7437180306751245, 'f1-score': 0.7448963258955668, 'support': 1582.0}, 'weighted avg': {'precision': 0.7477258763106639, 'recall': 0.7446270543615676, 'f1-score': 0.7458798374569908, 'support': 1582.0}}


Support: Support is the number of actual samples belonging to each class in the test dataset. In this experiment, the support values are 406, 390, 393, and 393 for classes 0, 1, 2, and 3 respectively. The total support is 1,582, which is the total number of test samples.

**Task 2 :**

In [9]:
np.random.seed(42)   # same starting weights as the model above, so only the ridge penalty differs

ridge_model = LogisticRegression(
    k=4,
    n=X_train_model.shape[1],
    method="batch",
    alpha=0.1,
    max_iter=2000,
    use_ridge=True,
    l=0.01,             # strength of the ridge penalty (lambda)
    verbose=False
)

ridge_model.fit(X_train_model, Y_train_onehot)

print("Ridge model training completed.")
print("Weight shape:", ridge_model.W.shape)

Ridge model training completed.
Weight shape: (47, 4)


In [15]:
# Predictions from the Ridge model
y_pred_ridge = ridge_model.predict(X_test_model)

print("Ridge Test Accuracy:", accuracy_score(y_test, y_pred_ridge))

print("\nRidge Classification Report:")
print(classification_report(y_test, y_pred_ridge, labels=[0, 1, 2, 3]))

Ridge Test Accuracy: 0.7149178255372945

Ridge Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       406
           1       0.62      0.56      0.59       390
           2       0.62      0.70      0.66       393
           3       0.82      0.77      0.79       393

    accuracy                           0.71      1582
   macro avg       0.72      0.71      0.71      1582
weighted avg       0.72      0.71      0.71      1582



Task 2 — Ridge/L2 Regularization:
Ridge regularization was added to the custom multinomial Logistic Regression with the option use_ridge=True (strength lambda, parameter l). It adds an L2 penalty on the weights (the bias is not penalised) to the loss and gradient, which discourages large weights. Using the same train/test split, the same random starting weights and the same hyperparameters (batch gradient descent, alpha=0.1, 2000 iterations), the model without regularization reached 74.46% test accuracy (macro-F1 0.745) and the Ridge model with lambda=0.01 reached 71.49% (macro-F1 0.71). So in this experiment Ridge did not help: with about 6,300 training samples and only 47 features the model does not overfit much, and the penalty mainly limits how well the weights can fit. In both models the middle price classes (1 and 2) are the hardest to separate (F1 about 0.6), while the cheapest and most expensive classes reach F1 of about 0.8. A proper search over lambda is done in the MLflow experiments in Task 3.

**Task 3**

In [17]:
import os
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# Where the experiments are logged. Local file for now (the CSIM server refuses my connection).
# If the server becomes reachable, only this line (and the login variables) has to change.
TRACKING_URI = "sqlite:///mlflow.db"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("st127171-a3")      # experiment name required by the assignment
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment  :", mlflow.get_experiment_by_name("st127171-a3").name)

Tracking URI: sqlite:///mlflow.db
Experiment  : st127171-a3


In [18]:
# MLflow needs a "flavor" to save a model. Ours is a hand-written class, so we wrap it in a pyfunc model.
# The wrapper keeps only the learned weights W and the fitted preprocessor, so anyone who loads it
# can pass RAW car columns (year, km_driven, brand, ...) and get a price class 0-3 back.
class CarPriceClassifier(mlflow.pyfunc.PythonModel):
    def __init__(self, W, preprocessor):
        self.W = W                        # shape (n_features + 1, 4), row 0 = bias
        self.preprocessor = preprocessor  # fitted ColumnTransformer (scaler + one-hot)

    def predict(self, context, model_input, params=None):
        X = self.preprocessor.transform(model_input)           # raw columns -> scaled / one-hot numbers
        X = np.column_stack([np.ones(len(X)), X])              # same intercept column as in training
        z = X @ self.W
        z = z - z.max(axis=1, keepdims=True)                   # stable softmax
        p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
        return np.argmax(p, axis=1)                            # class with the highest probability

e:\SABRINA AFRIN\anaconda py\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [19]:
# Validation split cut from TRAIN only: runs are compared on validation, the test set stays untouched for the final report.
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
X_tr_model  = add_intercept(preprocessor.transform(X_tr_raw))
X_val_model = add_intercept(preprocessor.transform(X_val_raw))
Y_tr_onehot = np.eye(4)[y_tr.astype(int)]

def metrics_of(m, y_true, y_hat, prefix):
    """All scores from OUR from-scratch functions, with a prefix (val_ or test_)."""
    return {
        f"{prefix}accuracy":           m.accuracy(y_true, y_hat),
        f"{prefix}macro_precision":    m.macro_precision(y_true, y_hat),
        f"{prefix}macro_recall":       m.macro_recall(y_true, y_hat),
        f"{prefix}macro_f1":           m.macro_f1(y_true, y_hat),
        f"{prefix}weighted_precision": m.weighted_precision(y_true, y_hat),
        f"{prefix}weighted_recall":    m.weighted_recall(y_true, y_hat),
        f"{prefix}weighted_f1":        m.weighted_f1(y_true, y_hat),
    }

# the grid: 3 GD methods x (no ridge + 3 lambdas) x 2 learning rates = 24 runs
grid = [(method, use_ridge, lam, alpha)
        for method in ["batch", "minibatch", "sto"]
        for use_ridge, lam in [(False, 0.0), (True, 0.001), (True, 0.01), (True, 0.1)]
        for alpha in [0.01, 0.1]]
MAX_ITER = 2000

for method, use_ridge, lam, alpha in grid:
    run_name = f"{method}-alpha{alpha}-" + (f"ridge{lam}" if use_ridge else "noridge")
    with mlflow.start_run(run_name=run_name):
        np.random.seed(42)            # same start weights / batches, so runs differ only by their settings
        m = LogisticRegression(k=4, n=X_tr_model.shape[1], method=method, alpha=alpha,
                               max_iter=MAX_ITER, use_ridge=use_ridge, l=lam, verbose=False)
        m.fit(X_tr_model, Y_tr_onehot)

        # 1) parameters
        mlflow.log_params({"method": method, "alpha": alpha, "max_iter": MAX_ITER,
                           "use_ridge": use_ridge, "lambda": lam,
                           "n_features": X_tr_model.shape[1], "n_classes": 4})

        # 2) metrics (validation is used to pick the best run, test is reported for reference)
        mlflow.log_metrics(metrics_of(m, y_val,  m.predict(X_val_model),  "val_"))
        mlflow.log_metrics(metrics_of(m, y_test, m.predict(X_test_model), "test_"))
        mlflow.log_metric("final_train_loss", float(m.losses[-1]))
        mlflow.log_metric("train_time_sec", float(m.train_time))
        for step in range(0, MAX_ITER, 100):          # loss curve, every 100 steps
            mlflow.log_metric("train_loss", float(m.losses[step]), step=step)

        # 3) the model itself (signature = column names/types only, NOT the dataset)
        wrapper = CarPriceClassifier(m.W, preprocessor)
        sig = infer_signature(X_val_raw.head(5), wrapper.predict(None, X_val_raw.head(5)))
        info = mlflow.pyfunc.log_model(name="model", python_model=wrapper, signature=sig,
                                       pip_requirements=["numpy", "pandas", "scikit-learn"])
        mlflow.set_tag("model_uri", info.model_uri)   # remember where the saved model is, used for registering it

    print(f"done: {run_name}")

2026/10/03 18:05:32 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


done: batch-alpha0.01-noridge


2026/10/03 18:05:37 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


done: batch-alpha0.1-noridge


2026/10/03 18:05:39 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


...

done: sto-alpha0.1-ridge0.01


2026/10/03 18:06:02 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


done: sto-alpha0.01-ridge0.1


2026/10/03 18:06:03 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


done: sto-alpha0.1-ridge0.1


In [20]:
exp = mlflow.get_experiment_by_name("st127171-a3")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["metrics.val_macro_f1 DESC"])

# the table is sorted by validation macro-F1, so row 0 is the best run
runs[["tags.mlflow.runName", "params.method", "params.alpha", "params.use_ridge", "params.lambda",
      "metrics.val_macro_f1", "metrics.val_accuracy", "metrics.test_macro_f1", "metrics.test_accuracy"]].head(10)

,tags.mlflow.runName,params.method,params.alpha,params.use_ridge,params.lambda,metrics.val_macro_f1,metrics.val_accuracy,metrics.test_macro_f1,metrics.test_accuracy
0,batch-alpha0.1-noridge,batch,0.1,False,0.0,0.724830,0.724901,0.740127,0.740202
1,minibatch-alpha0.1-noridge,minibatch,0.1,False,0.0,0.724717,0.724901,0.740720,0.740834
2,minibatch-alpha0.1-ridge0.001,minibatch,0.1,True,0.001,0.722292,0.722530,0.742183,0.742099
3,batch-alpha0.1-ridge0.001,batch,0.1,True,0.001,0.722256,0.722530,0.738450,0.738306
4,batch-alpha0.1-ridge0.01,batch,0.1,True,0.01,0.696212,0.698814,0.715561,0.716814
5,minibatch-alpha0.1-ridge0.01,minibatch,0.1,True,0.01,0.693784,0.697233,0.712145,0.713654
6,batch-alpha0.01-ridge0.01,batch,0.01,True,0.01,0.652832,0.658498,0.671240,0.673198
7,minibatch-alpha0.01-ridge0.01,minibatch,0.01,True,0.01,0.651856,0.657708,0.671240,0.673198
8,minibatch-alpha0.01-noridge,minibatch,0.01,False,0.0,0.646871,0.651383,0.668901,0.670038
9,batch-alpha0.01-noridge,batch,0.01,False,0.0,0.646871,0.651383,0.667634,0.668774


In [21]:
MODEL_NAME = "st127171-a3-model"          # name required by the assignment
client = MlflowClient()

best = runs.iloc[0]                       # row 0 = best validation macro-F1
print("Best run :", best["tags.mlflow.runName"], "| val macro-f1 =", round(best["metrics.val_macro_f1"], 4))

# put the saved model of that run into the Model Registry
mv = mlflow.register_model(model_uri=best["tags.model_uri"], name=MODEL_NAME)
print("Registered:", MODEL_NAME, "version", mv.version)

# "Staging": newer MLflow versions replaced the old stages with aliases, so set both (the old call may be unavailable)
client.set_registered_model_alias(MODEL_NAME, "staging", mv.version)
try:
    client.transition_model_version_stage(MODEL_NAME, mv.version, stage="Staging")
except Exception as e:
    print("Stage transition not available in this MLflow version, alias 'staging' is set instead:", type(e).__name__)

Best run : batch-alpha0.1-noridge | val macro-f1 = 0.7248
Registered: st127171-a3-model version 1


Successfully registered model 'st127171-a3-model'.
Created version '1' of model 'st127171-a3-model'.


In [22]:
# load the model by its registry name and the "staging" alias
staging_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@staging")

sample = X_test.head(5)                       # RAW columns (year, brand, ...), exactly what a user would send
print("Predicted classes:", staging_model.predict(sample))
print("True classes     :", y_test[:5])

Predicted classes: [1 2 1 0 0]
True classes     : [0 1 1 0 0]


In [23]:
import sys, joblib, sklearn

os.makedirs("app/model", exist_ok=True)

served = staging_model.unwrap_python_model()          # our CarPriceClassifier with W and the preprocessor
bundle = {
    "W": served.W,
    "preprocessor": served.preprocessor,
    "edges": df["selling_price"].quantile([0, .25, .5, .75, 1]).to_numpy(),   # the 4 price buckets (same quartiles as pd.qcut)
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    # value used by the web app when the user leaves a field empty
    "defaults": {**{c: float(X_train[c].median()) for c in numeric_features},
                 **{c: X_train[c].mode()[0] for c in categorical_features}},
}
joblib.dump(bundle, "app/model/car_model.pkl")

# same library versions inside Docker as in this notebook, otherwise the saved scaler/encoder may not load
with open("app/requirements.txt", "w") as f:
    f.write(f"dash\nnumpy=={np.__version__}\npandas=={pd.__version__}\n"
            f"scikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\n")

with open("app/Dockerfile", "w") as f:
    f.write(f"""FROM python:{sys.version_info.major}.{sys.version_info.minor}-slim
WORKDIR /code
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8050
CMD ["python", "app.py"]
""")
print("Saved app/model/car_model.pkl, app/requirements.txt and app/Dockerfile")
print("Price class edges:", bundle["edges"])

Saved app/model/car_model.pkl, app/requirements.txt and app/Dockerfile
Price class edges: [   29999.   270000.   450000.   690000. 10000000.]
